# Silver layer transformaion

### Decisions made
- Dropped rows where `Event distance/length` contains `d` (days), as invalid per lab spec.
- Dropped rows where `Athlete performance` contains `d`, same reason.
- Dropped rows where event unit and performance unit match (`km/km`, `mi/mi`, `h/h`) because it indicates mismatched data.
- Dropped rows where Event distance/length == `mi` (missing number)
- Dropped null values in `Event distance/length` and `Athlete performance` since rows with null distance or performance can't contribute to any gold metric.
- Rows where `Event distance/length` ends with `k` converted to `km` before filtering.
- `Athlete performance` converted to decimal hours (distance races) or decimal km (timed races).
- `event_id` created with `sha2` on event name because `dense_rank` didn't work for streaming table.
- `athlete_id` created with `sha2` on Athlete ID to enable a `dim_athlete` table in gold.
- Column names converted to `snake_case`
- `athlete_country` = `"XXX"` replaced with null
- Clubs with `*`-prefix: stripped with `regexp_replace`
- Dates standardized: start date extracted from interval format (`09.-10.02.2008` -> `09.02.2008`), parsed to `DateType` with format `dd.MM.yyyy`
- Table name: `marathos.silver.races_clean_obt`
- `Miles`/`Mile`/`Miles` -> `mi`
- Dropped relay races (`4x52 km` via x-pattern)
- Dropped distances > `500`
- Dropped rows with `/` or `,` in `Event distance/length`
- `event_unit` extracted as separate column (`km`, `mi`, `h`)
- `Event distance/length` cast to `double`
- Gender: `M` -> `Male`, `F` -> `Female`, `X` -> `Other`, `null` -> `Other`
- Year of birth < `1700` or > `2005` -> `null`
- `Athlete country` uppercased
- `Athlete club` null -> `"Unknown"`
- Athlete average speed > `50` dropped
- `event_country` extracted via regex on Event name
- stockholm_marathon_2024.csv added to volume
- mi -> km conversion (* 1.60934) + event_unit updated to km

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, when, to_date, regexp_replace
from pyspark.sql import functions as F
from utils.utils import rename_columns_to_snake_case

@dp.table(
    name="marathos.silver.races_clean_obt",
    comment="Cleaned race data",
    table_properties={
        "delta.columnMapping.mode": "name",
        "delta.minReaderVersion": "2",
        "delta.minWriterVersion": "5",
    },
)
def races_clean_obt():
    df = dp.read_stream("marathos.bronze.races")

    # Standardize values before filtering
    df = (df
        .withColumn("Event name", F.regexp_replace(col("Event name"), r"\s*\(\)$", ""))
        .withColumn("Athlete country", F.upper(col("Athlete country")))
        .withColumn("Athlete club", F.coalesce(col("Athlete club"), F.lit("Unknown")))
        .withColumn(
            "Event distance/length",
            when(col("Event distance/length").like("%miles%"), "mi")
            .when(col("Event distance/length").like("%Mile%"), "mi")
            .when(col("Event distance/length").like("%Miles%"), "mi")
            .otherwise(F.regexp_replace(col("Event distance/length"), "k$", "km"))
        )
    )

    # Filter out invalid distances: bare units, relays, days, unit mismatches
    df = (df
        .filter(col("Event distance/length") != "mi")
        .filter(~col("Event distance/length").rlike(r"\d+x\d+"))
        # LLM — extract numeric part to compare against upper bound
        .filter(F.regexp_extract(col("Event distance/length"), r"^(\d+\.?\d*)", 1).cast("double") <= 500)
        .filter(~col("Event distance/length").contains("/"))
        .filter(~col("Event distance/length").contains(","))
        .filter(~col("Event distance/length").like("%d%"))
        .filter(~col("Athlete performance").like("%d%"))
        .filter(
            ~(col("Event distance/length").like("%km") & col("Athlete performance").like("%km"))
            & ~(col("Event distance/length").like("%mi") & col("Athlete performance").like("%mi"))
            & ~(col("Event distance/length").like("%h") & col("Athlete performance").like("%h"))
        )
    )

    # Extract event_unit, cast distance to double, convert mi to km
    df = (df
        .withColumn(
            "event_unit",
            when(col("Event distance/length").like("%km"), "km")
            .when(col("Event distance/length").like("%mi"), "mi")
            .when(col("Event distance/length").like("%h"), "h")
        )
        # LLM — extract numeric part and cast to double, unit already captured in event_unit
        .withColumn(
            "Event distance/length",
            F.regexp_extract(col("Event distance/length"), r"^(\d+\.?\d*)", 1).cast("double")
        )
        .withColumn(
            "Event distance/length",
            when(col("event_unit") == "mi", col("Event distance/length") * 1.60934)
            .otherwise(col("Event distance/length"))
        )
        .withColumn(
            "event_unit",
            when(col("event_unit") == "mi", F.lit("km")).otherwise(col("event_unit"))
        )
    )

    # Expand gender codes: M -> Male, F -> Female, X -> Other
    df = df.withColumn(
        "Athlete gender",
        when(col("Athlete gender") == "M", "Male")
        .when(col("Athlete gender") == "F", "Female")
        .when(col("Athlete gender") == "X", "Other")
        .otherwise("Other")
    )

    # Nullify year of birth outside reasonable range because minimum age 18 to compete
    df = df.withColumn(
        "Athlete year of birth",
        when(col("Athlete year of birth") < 1700, None)
        .when(col("Athlete year of birth") > 2005, None)
        .otherwise(col("Athlete year of birth"))
    )

    # Convert athlete performance to decimal:
    # distance races (km): h:mm:ss -> decimal hours
    # timed races (h): strip unit, cast to decimal km
    df = (df
        .withColumn("Athlete performance", F.regexp_replace("Athlete performance", "h", ""))
        .withColumn("performance_split", F.split(F.col("Athlete performance"), ":"))
        .withColumn(
            "Athlete performance",
            when(
                col("event_unit") == "h",
                F.regexp_replace(col("Athlete performance"), " km", "").cast("double"),
            ).otherwise(
                F.col("performance_split")[0].cast("double")
                + F.col("performance_split")[1].cast("double") / 60
                + F.col("performance_split")[2].cast("double") / 3600
            ),
        )
    )

    # Drop impossible speeds and replace unknown country code with null
    df = (df
        .filter(col("Athlete average speed").isNull() | (col("Athlete average speed") <= 50))
        .withColumn(
            "Athlete country",
            when(col("Athlete country") == "XXX", None).otherwise(col("Athlete country")),
        )
    )

    # LLM generated — three regex steps extract start date from interval formats before parsing to DateType
    df = (df
        .withColumn("Event dates", F.regexp_replace(col("Event dates"), r"(\d{2}\.\d{2}\.)-\d{2}\.\d{2}\.(\d{4})", "$1$2"))
        .withColumn("Event dates", F.regexp_replace(col("Event dates"), r"(\d{2}\.\d{2}\.\d{4})-\d{2}\.\d{2}\.\d{4}", "$1"))
        .withColumn("Event dates", F.regexp_replace(col("Event dates"), r"(\d{2})\.-\d{2}\.", "$1."))
        .withColumn("Event dates", F.to_date(col("Event dates"), "dd.MM.yyyy"))
    )

    # Generate surrogate keys with sha2 (LLM), extract event country from event name,
    # drop rows missing distance or performance
    df = (df
        .withColumn("Athlete club", F.regexp_replace("Athlete club", "\\*", ""))
        .withColumn("event_id", F.sha2(col("Event name"), 256))
        .withColumn("athlete_id", F.sha2(col("Athlete ID").cast("string"), 256))
        .withColumn(
            "event_country",
            when(F.regexp_extract(col("Event name"), r"\(([A-Z]{3})\)", 1) == "", None)
            .otherwise(F.regexp_extract(col("Event name"), r"\(([A-Z]{3})\)", 1))
        )
        .dropna(subset=["Event distance/length", "Athlete performance"])
        .drop("performance_split", "Athlete ID")
    )

    return rename_columns_to_snake_case(df)